# CNN desde cero vs MobileNetV2 · reentrenamiento corregido (Práctica 2)

En la corrida anterior (`resultados/`) la CNN desde cero sacó **41.8 %** y MobileNetV2 **93.6 %**. La arquitectura de la
CNN está bien; falló el entrenamiento:

- `ReduceLROnPlateau` bajó la tasa de 1e-3 a 7.8e-6 antes de la época 46, porque la validación salió casi al azar en
  las primeras épocas (BatchNorm con `momentum=0.99` tarda en estabilizarse).
- Terminó en la época 60 **todavía subiendo** y con train ≈ val ≈ 43 %: le faltó entrenar, no se sobreajustó.

Aquí se entrena **la misma arquitectura** con:

| Cambio | Antes | Ahora |
|---|---|---|
| Tasa de aprendizaje | `ReduceLROnPlateau` | calentamiento 3 épocas + **coseno** fijo (no depende de val) |
| Épocas | 60 | **120** (se guarda la mejor en val) |
| BatchNorm | `momentum=0.99` | `momentum=0.9` |

MobileNetV2 se entrena con su receta de antes (3 épocas solo la capa nueva, luego toda la red), así ambas usan la
**misma división**. Las salidas tienen el mismo formato que `resultados/`.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo → GPU (T4)`, y en Drive, si `escom_caras` está en
*Compartido conmigo*: clic derecho → **Organizar → Agregar acceso directo → Mi unidad**.

## 1. Configuración

In [ ]:
# 'aleatoria' = 20 % de las fotos al azar, como la corrida anterior (comparable con resultados/).
# 'por_set'   = sesiones completas en test: ropa, fondo y luz que el modelo no vio (más honesta, más difícil).
DIVISION = 'aleatoria'   #@param ['aleatoria', 'por_set']
ENTRENAR_SCRATCH   = True   #@param {type:"boolean"}
ENTRENAR_MOBILENET = True   #@param {type:"boolean"}

EPOCAS_SCRATCH = 120        #@param {type:"integer"}
EPOCAS_MOBILENET_CABEZA, EPOCAS_MOBILENET_TODA = 3, 22
LOTE, SEMILLA = 64, 42
FRAC_TEST, FRAC_VAL = 0.20, 0.15

CARPETA_DRIVE = ''   # vacío = se busca sola la carpeta que tenga caras_manifest.csv dentro de MyDrive

## 2. Entorno y datos

In [ ]:
import os, json, time, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print('TF', tf.__version__, '| Keras', keras.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
keras.mixed_precision.set_global_policy('mixed_float16')   # ~2x más rápido en la T4
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

from google.colab import drive
drive.mount('/content/drive')
if CARPETA_DRIVE:
    RAIZ = Path(CARPETA_DRIVE)
else:
    directo = Path('/content/drive/MyDrive/escom_caras/caras_manifest.csv')
    candidatos = [directo] if directo.exists() else         sorted(Path('/content/drive/MyDrive').glob('**/caras_manifest.csv'), key=lambda p: len(p.parts))
    assert candidatos, 'No encontré caras_manifest.csv en MyDrive: agrega el acceso directo a escom_caras o llena CARPETA_DRIVE'
    RAIZ = candidatos[0].parent
SALIDA = RAIZ / f'resultados_v2_{DIVISION}'
SALIDA.mkdir(exist_ok=True)
print('Datos:', RAIZ, '| salidas:', SALIDA)

In [ ]:
# Copia local de las caras: leer 29 mil archivos del Drive en cada época sería lentísimo.
manifest = pd.read_csv(RAIZ / 'caras_manifest.csv')
ORIGEN, LOCAL = RAIZ / 'caras', Path('/content/caras')

def copiar(ruta):
    destino = LOCAL / ruta
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(ORIGEN / ruta, destino)

t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(copiar, manifest['ruta']), 1):
        if i % 5000 == 0:
            print(f'{i}/{len(manifest)} ({time.time() - t0:.0f} s)')
print(f'{len(manifest)} caras copiadas en {time.time() - t0:.0f} s')

## 3. División

- `aleatoria`: estratificada por persona, como antes. Fotos de la misma sesión quedan en train y en test.
- `por_set`: por persona, se eligen sets completos al azar (con semilla) hasta juntar ~20 % para test y ~15 % para val.

In [ ]:
CLASES = sorted(manifest['persona'].unique())
A_INDICE = {p: i for i, p in enumerate(CLASES)}
manifest['y'] = manifest['persona'].map(A_INDICE)

if DIVISION == 'aleatoria':
    resto, test = train_test_split(manifest, test_size=FRAC_TEST, stratify=manifest['y'], random_state=SEMILLA)
    train, val = train_test_split(resto, test_size=FRAC_VAL / (1 - FRAC_TEST), stratify=resto['y'], random_state=SEMILLA)
    manifest['split'] = 'train'
    manifest.loc[val.index, 'split'] = 'val'
    manifest.loc[test.index, 'split'] = 'test'
else:
    rng = np.random.default_rng(SEMILLA)
    manifest['split'] = 'train'
    for persona, g in manifest.groupby('persona'):
        tam = g.groupby('set').size()
        sets = list(rng.permutation(tam.index))
        for nombre, frac in [('test', FRAC_TEST), ('val', FRAC_VAL)]:
            juntadas = 0
            while juntadas < frac * len(g) and len(sets) > 2:
                s = sets.pop(0); juntadas += tam[s]
                manifest.loc[(manifest['persona'] == persona) & (manifest['set'] == s), 'split'] = nombre
    assert manifest.groupby(['persona', 'set'])['split'].nunique().max() == 1

print(manifest['split'].value_counts().to_dict())
pd.crosstab(manifest['persona'], manifest['split']).T

In [ ]:
N_CLASES = len(CLASES)
AUTO = tf.data.AUTOTUNE

def dataset(split, lado, entrenar):
    d = manifest[manifest['split'] == split]
    rutas = [str(LOCAL / r) for r in d['ruta']]
    ds = tf.data.Dataset.from_tensor_slices((rutas, d['y'].to_numpy()))
    def leer(ruta, y):
        img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
        img = tf.image.resize(img, (lado, lado), antialias=True)
        return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8), y
    # uint8 en caché: en RAM a 128 px (~1.4 GB), en disco a 224 px (~4.4 GB no caben junto con lo demás)
    ds = ds.map(leer, num_parallel_calls=AUTO).cache('' if lado <= 128 else f'/content/cache_{split}_{lado}')
    if entrenar:
        ds = ds.shuffle(8192, seed=SEMILLA, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), tf.one_hot(y, N_CLASES)), num_parallel_calls=AUTO)
    return ds.batch(LOTE, drop_remainder=entrenar).prefetch(AUTO)


def aumento():
    # El mismo aumento que la corrida anterior (sale de su config.json)
    return keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.2, value_range=(0, 255)),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
    ], name='aumento')

## 4. Modelos

In [ ]:
def cnn_scratch(lado=128, momentum_bn=0.9):
    # Misma arquitectura que keras_scratch.keras; solo cambia el momentum de BatchNorm (0.99 -> 0.9)
    entrada = keras.Input((lado, lado, 3))
    x = aumento()(entrada)
    x = layers.Rescaling(1 / 255)(x)
    for filtros, drop in [(32, 0.10), (64, 0.15), (128, 0.20), (256, 0.25)]:
        for _ in range(2):
            x = layers.Conv2D(filtros, 3, padding='same', use_bias=False)(x)
            x = layers.BatchNormalization(momentum=momentum_bn)(x)
            x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    salida = layers.Dense(N_CLASES, activation='softmax', dtype='float32')(x)
    return keras.Model(entrada, salida, name='keras_scratch_v2')


def mobilenet(lado=224):
    base = keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(lado, lado, 3))
    entrada = keras.Input((lado, lado, 3))
    x = aumento()(entrada)
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(N_CLASES, activation='softmax', dtype='float32')(x)
    return keras.Model(entrada, salida, name='keras_mobilenetv2_v2'), base


def coseno(pico, epocas, pasos_por_epoca, calentamiento=3):
    return keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=pico / 20, warmup_target=pico, warmup_steps=calentamiento * pasos_por_epoca,
        decay_steps=(epocas - calentamiento) * pasos_por_epoca, alpha=0.01)


def compilar(modelo, lr):
    modelo.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
                   loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1), metrics=['accuracy'])


def ajustar(modelo, nombre, epocas, ds_train, ds_val, epoca_inicial=0, mejor_previo=None):
    carpeta = SALIDA / nombre; carpeta.mkdir(exist_ok=True)
    # mejor_previo: en la fase 2 solo guarda si supera a la mejor época de la fase 1
    mejor = keras.callbacks.ModelCheckpoint(str(carpeta / 'mejor.weights.h5'), monitor='val_accuracy', mode='max',
                                            save_best_only=True, save_weights_only=True,
                                            initial_value_threshold=mejor_previo)
    h = modelo.fit(ds_train, validation_data=ds_val, epochs=epoca_inicial + epocas, initial_epoch=epoca_inicial,
                   callbacks=[mejor], verbose=2)
    return pd.DataFrame(h.history).assign(epoca=np.arange(epoca_inicial + 1, epoca_inicial + epocas + 1))

## 5. Evaluación (mismo formato que `resultados/`)

In [ ]:
def evaluar(modelo, nombre, ds_test, historial, segundos):
    carpeta = SALIDA / nombre
    modelo.load_weights(str(carpeta / 'mejor.weights.h5'))
    test = manifest[manifest['split'] == 'test']
    y_real = test['y'].to_numpy()
    y_pred = modelo.predict(ds_test, verbose=0).argmax(1)
    met = dict(modelo=nombre, division=DIVISION, n_prueba=int(len(y_real)), n_personas=N_CLASES,
               accuracy=round(accuracy_score(y_real, y_pred), 4),
               f1_macro=round(f1_score(y_real, y_pred, average='macro'), 4),
               f1_ponderado=round(f1_score(y_real, y_pred, average='weighted'), 4),
               mejor_epoca=int(historial.loc[historial['val_accuracy'].idxmax(), 'epoca']),
               minutos_entrenamiento=round(segundos / 60, 1))
    (carpeta / 'metricas.json').write_text(json.dumps(met, indent=2, ensure_ascii=False))
    historial.to_csv(carpeta / 'historial.csv', index=False)
    pd.DataFrame(classification_report(y_real, y_pred, target_names=CLASES, output_dict=True)).T.iloc[:N_CLASES] \
        .to_csv(carpeta / 'por_clase.csv')
    pd.DataFrame(confusion_matrix(y_real, y_pred), index=CLASES, columns=CLASES).to_csv(carpeta / 'matriz_confusion.csv')
    mal = y_pred != y_real
    pd.DataFrame(dict(ruta=test['ruta'].to_numpy()[mal], real=[CLASES[i] for i in y_real[mal]],
                      predicho=[CLASES[i] for i in y_pred[mal]])).to_csv(carpeta / 'errores.csv', index=False)
    modelo.save(str(carpeta / f'{nombre}.keras'))

    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m, titulo in [(ejes[0], 'loss', 'Pérdida'), (ejes[1], 'accuracy', 'Accuracy')]:
        ax.plot(historial['epoca'], historial[m], 'r', label='train')
        ax.plot(historial['epoca'], historial['val_' + m], 'g', label='validación')
        ax.axvline(met['mejor_epoca'], ls=':', c='b', label=f"mejor época = {met['mejor_epoca']}")
        ax.set_title(titulo); ax.set_xlabel('época'); ax.legend(); ax.grid(alpha=0.3)
    plt.suptitle(nombre); plt.tight_layout(); plt.savefig(carpeta / 'curvas.png', dpi=110); plt.show()
    print(json.dumps(met, indent=2, ensure_ascii=False))
    return met

RESULTADOS = {}

## 6. CNN desde cero (≈ 25-40 min en T4)

In [ ]:
if ENTRENAR_SCRATCH:
    tr, va, te = dataset('train', 128, True), dataset('val', 128, False), dataset('test', 128, False)
    pasos = len(manifest[manifest['split'] == 'train']) // LOTE
    modelo = cnn_scratch()
    compilar(modelo, coseno(1e-3, EPOCAS_SCRATCH, pasos))
    print(f'{modelo.count_params() / 1e6:.2f} M parámetros')
    t0 = time.time()
    hist = ajustar(modelo, 'keras_scratch_v2', EPOCAS_SCRATCH, tr, va)
    RESULTADOS['keras_scratch_v2'] = evaluar(modelo, 'keras_scratch_v2', te, hist, time.time() - t0)
    del modelo; keras.backend.clear_session()

## 7. MobileNetV2 con la misma división (≈ 35-45 min en T4)

In [ ]:
if ENTRENAR_MOBILENET:
    tr, va, te = dataset('train', 224, True), dataset('val', 224, False), dataset('test', 224, False)
    pasos = len(manifest[manifest['split'] == 'train']) // LOTE
    modelo, base = mobilenet()
    t0 = time.time()
    base.trainable = False                       # fase 1: solo la capa nueva
    compilar(modelo, 1e-3)
    h1 = ajustar(modelo, 'keras_mobilenetv2_v2', EPOCAS_MOBILENET_CABEZA, tr, va)
    base.trainable = True                        # fase 2: toda la red, tasa baja con coseno
    compilar(modelo, coseno(1e-4, EPOCAS_MOBILENET_TODA, pasos, calentamiento=1))
    h2 = ajustar(modelo, 'keras_mobilenetv2_v2', EPOCAS_MOBILENET_TODA, tr, va, epoca_inicial=EPOCAS_MOBILENET_CABEZA,
                 mejor_previo=float(h1['val_accuracy'].max()))
    hist = pd.concat([h1, h2], ignore_index=True)
    RESULTADOS['keras_mobilenetv2_v2'] = evaluar(modelo, 'keras_mobilenetv2_v2', te, hist, time.time() - t0)
    del modelo; keras.backend.clear_session()

## 8. Comparación con la corrida anterior

In [ ]:
anteriores = {   # resultados/<modelo>/metricas.json de la corrida anterior (división aleatoria)
    'keras_scratch (antes)':     dict(division='aleatoria', accuracy=0.4181, f1_macro=0.3818, mejor_epoca=60),
    'keras_mobilenetv2 (antes)': dict(division='aleatoria', accuracy=0.9359, f1_macro=0.9347, mejor_epoca=25),
}
tabla = pd.DataFrame({**anteriores, **RESULTADOS}).T[['division', 'accuracy', 'f1_macro', 'mejor_epoca', 'minutos_entrenamiento']]
tabla.to_csv(SALIDA / 'comparacion.csv')
display(tabla)
if DIVISION == 'por_set':
    print('Ojo: la corrida anterior usó división aleatoria; compara "antes" contra "v2" solo como referencia.')

!cd "{SALIDA.parent}" && zip -qr /content/{SALIDA.name}.zip {SALIDA.name} -x "*.keras" "*.h5"
from google.colab import files
files.download(f'/content/{SALIDA.name}.zip')   # sin pesos (pesan mucho); los pesos se quedan en el Drive